# Fine-tune CENTINAL's CSRNet on a Colab GPU

Trains the crowd-counting model on ShanghaiTech Part A + Part B, starting from the
bundled `csrnet_shanghai.pth`, and saves the best checkpoint to your Google Drive.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then run the cells top to bottom.

Checkpoints go to Drive, not the Colab disk, because free Colab sessions can
disconnect at any time and the local disk is wiped when they do. If you get
disconnected, rerun cells 1–4 and then use the **Resume** cell.

## 1. Settings

In [ ]:
# Where the code comes from. "github" clones the branch; "upload" expects you to
# upload centinal_colab_bundle.zip (see the README section of cell 3).
CODE_SOURCE = "github"
REPO_URL = "https://github.com/Daksh-Upadhayay/CENTINAL.git"
BRANCH = "improve-models-and-eval"

# Training. The local run swung a lot between epochs at 1e-5 and barely moved at
# 1e-6, so 3e-6 is a starting point between the two -- not a tuned value.
LR = 3e-6
EPOCHS = 60
EVAL_EVERY = 2            # evaluating both test splits costs roughly as much as a training epoch
TIME_BUDGET_MIN = 150     # stop cleanly before a long free-tier session is likely to be cut off

DRIVE_DIR = "/content/drive/MyDrive/centinal_training"
CHECKPOINT = f"{DRIVE_DIR}/csrnet_centinal.pth"

## 2. Check the GPU and mount Drive

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun this cell.")
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Checkpoints will be saved to", DRIVE_DIR)

## 3. Get the code

**`github`** clones the branch. It must be pushed first:
`git push -u origin improve-models-and-eval`

**`upload`** skips GitHub. On your Mac, from the repo folder, run:

```
zip -r centinal_colab_bundle.zip centinal train_csrnet.py eval/eval_csrnet.py csrnet_shanghai.pth
```

Then set `CODE_SOURCE = "upload"` in cell 1 and run this cell. It will prompt for the file.

In [ ]:
import os
%cd /content
if CODE_SOURCE == "github":
    !rm -rf /content/CENTINAL
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/CENTINAL
else:
    from google.colab import files
    uploaded = files.upload()   # choose centinal_colab_bundle.zip
    !rm -rf /content/CENTINAL && mkdir -p /content/CENTINAL
    !unzip -q -o centinal_colab_bundle.zip -d /content/CENTINAL

# Stop here if the code did not arrive, instead of failing later with a
# confusing "file not found" from the training cell.
if not os.path.exists("/content/CENTINAL/train_csrnet.py"):
    raise RuntimeError(
        "Code not found in /content/CENTINAL. If CODE_SOURCE is 'github', the branch "
        f"'{BRANCH}' is probably not pushed yet -- run `git push -u origin {BRANCH}` "
        "on your Mac, then rerun this cell. Otherwise use CODE_SOURCE = 'upload'.")

%cd /content/CENTINAL
!ls train_csrnet.py centinal/ csrnet_shanghai.pth

## 4. Download ShanghaiTech (~174 MB)

In [ ]:
import os
if not os.path.isdir("/content/data/part_A_final"):
    !curl -L --fail -o /content/shanghaitech.zip "https://www.dropbox.com/scl/fi/dkj5kulc9zj0rzesslck8/ShanghaiTech_Crowd_Counting_Dataset.zip?rlkey=ymbcj50ac04uvqn8p49j9af5f&dl=1"
    !unzip -q -o /content/shanghaitech.zip -d /content/data
!for p in A B; do for s in train test; do echo "part_$p $s: $(ls /content/data/part_${p}_final/${s}_data/images | wc -l) images"; done; done

## 5. Train

The first thing the script prints is **epoch 0**: the starting checkpoint's score
before any training. It should be close to **Part A MAE 136.7, Part B MAE 14.1**.
If it isn't, stop the cell. That means the setup disagrees with how the
checkpoint was trained, and training would make the model worse.

A checkpoint is only saved when it beats the best score so far, where the score
is the mean of each split's MAE divided by its average head count. So whatever
ends up on Drive is at least as good as what you started with.

In [ ]:
%cd /content/CENTINAL
!python -u train_csrnet.py \
    --data_root /content/data \
    --part AB \
    --preprocess raw \
    --init_from csrnet_shanghai.pth \
    --lr {LR} --epochs {EPOCHS} --eval_every {EVAL_EVERY} \
    --time_budget_min {TIME_BUDGET_MIN} \
    --out {CHECKPOINT}

### Resume (only if the session disconnected)

Rerun cells 1–4 first, then this cell. It continues from the best checkpoint on
Drive. Its epoch 0 will show that checkpoint's score rather than 136.7 / 14.1,
which is expected. The optimiser restarts from scratch, which is fine for fine-tuning.

In [ ]:
%cd /content/CENTINAL
!python -u train_csrnet.py \
    --data_root /content/data \
    --part AB \
    --preprocess raw \
    --init_from {CHECKPOINT} \
    --lr {LR} --epochs {EPOCHS} --eval_every {EVAL_EVERY} \
    --time_budget_min {TIME_BUDGET_MIN} \
    --out {CHECKPOINT}

## 6. Evaluate the best checkpoint

Runs the same benchmark script used for the README, so the numbers are directly
comparable. The bundled model's scores are:

| | Part A MAE | Part B MAE |
|---|---|---|
| `csrnet_shanghai.pth` (original) | 136.60 | 14.28 |
| `csrnet_centinal.pth` (current default) | 72.98 | 13.38 |
| published CSRNet | 68.2 | 10.6 |

A new model is only worth switching to if it beats the current default on Part A
**without** giving up much on Part B.

In [ ]:
%cd /content/CENTINAL
import torch
meta = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
print("Best checkpoint is from epoch", meta["epoch"])

for part in ("A", "B"):
    !python eval/eval_csrnet.py \
        --model_path {CHECKPOINT} \
        --dataset_path /content/data/part_{part}_final/test_data \
        --tag part{part}_finetuned \
        --output_dir {DRIVE_DIR}/results

## 7. Bring it back

Download these from `MyDrive/centinal_training/` in Google Drive:

- `csrnet_centinal.pth`: put it in the repo root, replacing the old one
- `csrnet_centinal_history.json`: per-epoch scores
- `results/`: per-image CSVs and scatter plots

Then ask Claude to compare it against the current default. If it's better, the
risk classifier needs retraining too, because its feature scaling was calibrated
on the current CSRNet.